In [8]:
%load_ext autoreload
%autoreload 2
from sindex.sources.datacite.utils import get_relevant_citations_block_from_ndjson
from sindex.sources.datacite.jobs import (
    batch_slim_datacite_record_to_ndjson_fast,
    batch_find_citations_dc_from_citation_block,
batch_find_citations_dc_from_citation_block_optimized,
extract_unique_dois_from_citation_blocks,
lookup_dates_in_oa_snapshot,
batch_find_citations_from_dc_parallel,
)
import duckdb
from pathlib import Path
import os

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Slim datacite citations raw ndjson

In [5]:
src_folder = r"I:\pipeline-data\citations\datacite\datacite-raw-with-citations"
dst_folder = r"I:\pipeline-data\citations\datacite\datacite-slim-with-citations"
summary = batch_slim_datacite_record_to_ndjson_fast(
    src_folder = str(src_folder),
    dst_folder = str(dst_folder),
    overwrite = False,          # overwrite for repeatable tests
    accept_gz = True,
    one_line_progress = True
)

# Summary
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

print("\nCompleted.")

Processing 769 files using 40 cores...
[769/769] files completed
Done. files=769 kept=20,147,279 bad=0 time=3665.1s rate≈5,496/rec-per-sec

Summary:
  files_seen: 769
  records_read: 20147279
  records_kept: 20147279
  records_bad_json: 0
  output_dir: \\192.168.20.168\AILarge\pipeline-data\citations\datacite\datacite-slim-with-citations
  elapsed_sec: 3665.14
  rate_rec_per_sec: 5496

Completed.


In [3]:
def count_lines(directory):
    total_lines = 0
    for filename in os.listdir(directory):
        if filename.endswith(".ndjson"):
            path = os.path.join(directory, filename)
            with open(path, 'rb') as f:
                count = sum(1 for line in f)
                total_lines += count
    return total_lines
slim_path = r"D:\pipeline-data\citations\datacite\datacite-slim-with-citations"
print(f"Total: {count_lines(slim_path)}")

Total: 48964382


## Save citations block matching to our DOIs

In [4]:
get_relevant_citations_block_from_ndjson(
     db_path = r"D:\pipeline-data\records\slim-records\datacite-slim-records.duckdb",
     ndjson_folder = r"D:\pipeline-data\citations\datacite\datacite-slim-with-citations",
     target_table = "my_datasets",
     output_file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson",
     reset_log = True
 )

[2026-01-24 23:01:47.623156] Batch processing 769 files...
[2026-01-24 23:02:56.718252] Complete!
Total DOIs Matched:       48,961,051
Total Records Saved:      1,565,793


In [6]:
def count_lines(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        count = 0
        for _ in f:
            count += 1
    return count
file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
print(f"Total objects: {count_lines(file_path):,}")

Total objects: 1,565,793


## Get Datacite citations

In [24]:
folder_path = r"D:\pipeline-data\citations\datacite\citation_blocks"
out_path = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
oa_db_path = r"D:\pipeline-data\external\openalex-snapshot\duckdb\oa_snapshot.duckdb"

In [25]:
batch_find_citations_dc_from_citation_block_optimized(folder_path, out_path, oa_db_path)

[*] Found 1 file(s). Connecting to DuckDB...

--- Processing: citation_blocks.ndjson (1/1) ---
[21:11:54] Step 1: Extracting DOIs and loading file into memory...
    -> Extracted 1,565,793 rows and 896,417 unique citation DOIs.
['10.1021/acs.inorgchem.7b01865', '10.1186/s13059-025-03683-7', '10.1103/physrevd.95.012010', '10.1021/ic500290m', '10.1016/j.physletb.2009.10.050', '10.1016/s0022-328x(02)01690-x', '10.1021/ja111504w', '10.15468/dl.yja0eg', '10.1080/25739638.2024.2433912', '10.57451/lhd.a.gas_puf.185859.1']
[21:12:24] Step 2: Querying DuckDB for 896,417 DOIs...


OutOfMemoryException: Out of Memory Error: failed to pin block of size 256.0 KiB (99.9 GiB/100.0 GiB used)

Possible solutions:
* Reducing the number of threads (SET threads=X)
* Disabling insertion-order preservation (SET preserve_insertion_order=false)
* Increasing the memory limit (SET memory_limit='...GB')

See also https://duckdb.org/docs/stable/guides/performance/how_to_tune_workloads

### Extract unique DOIs from DataCite citations

In [32]:
# Extract unique DOIs from DataCite citations
folder_path = r"D:\pipeline-data\citations\datacite\citation_blocks"
output_parquet = r"D:\pipeline-data\citations\datacite\datacite_unique_citation_doi.parquet"
extract_unique_dois_from_citation_blocks(folder_path, output_parquet)

[*] Scanning 1 files in citation_blocks...
    -> Finished citation_blocks.ndjson. Current unique DOIs: 896,417

[*] Final Save: Exporting 896,417 unique DOIs to D:\pipeline-data\citations\datacite\datacite_unique_citation_dois...
[SUCCESS] Aggregate DOI list saved to D:\pipeline-data\citations\datacite\datacite_unique_citation_dois


In [34]:
duckdb.execute(f"SELECT doi FROM read_parquet('{output_parquet}') LIMIT 5").df()

,doi
0,10.1021/acs.inorgchem.7b01865
1,10.1186/s13059-025-03683-7
2,10.1103/physrevd.95.012010
3,10.1021/ic500290m
4,10.1016/j.physletb.2009.10.050


### Find publication dates for these DOIs in the OpenAlex Snapshot

In [11]:
# Paths
input_parquet = r"D:\pipeline-data\citations\datacite\datacite_unique_citation_dois.parquet"
output_parquet = r"D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"
oa_db_path = r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb"

In [15]:
# Run
lookup_dates_in_oa_snapshot(oa_db_path, input_parquet, output_parquet)

Joining D:\pipeline-data\citations\datacite\datacite_unique_citation_dois with OpenAlex database...
    [SUCCESS] Found 884,723 matches.
    [INFO] Results saved to: D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates
    [INFO] Time taken: 41.88s


In [16]:
# View
duckdb.execute(f"SELECT* FROM read_parquet('{output_parquet}') LIMIT 5").df()

,doi,pubdate
0,10.1021/acs.inorgchem.6b01545,2016-09-27
1,10.12688/f1000research.2-159.v1,2013-07-17
2,10.1257/aer.20210413,2023-05-31
3,10.1080/2150704x.2016.1234726,2016-09-29
4,10.1159/000540058,2024-06-27


In [17]:
#Check
con = duckdb.connect(r"C:\Users\BPatel\Documents\oa_duckdb_fast\oa_snapshot.duckdb")

# Replace with the DOI you want to check
test_doi = '10.1257/aer.20210413'

# Direct lookup
result = con.execute("SELECT doi, pubdate FROM openalex_pubdate WHERE doi = ?", [test_doi]).df()
print(result)
con.close()

                    doi     pubdate
0  10.1257/aer.20210413  2021-03-01
1  10.1257/aer.20210413  2023-05-31


### Create citations files

In [4]:
folder_path = r"D:\pipeline-data\citations\datacite\citation_blocks"
out_ndjson = r"D:\pipeline-data\citations\datacite\dc_citations.ndjson"
date_parquet = r"D:\pipeline-data\citations\datacite\datacite_citation_dois_with_pubdates.parquet"

In [25]:
batch_find_citations_dc_from_citation_block_optimized(folder_path, out_ndjson, date_parquet)

[*] Loading pubdate cache from datacite_citation_dois_with_pubdates.parquet...
[*] Calculating total workload (scanning line counts)...
    -> Ready to process 1,565,793 lines across 1 files.
Progress: 3.13% | Line 48,951/1,565,793 | File 1/1 | Citations: 100,650

KeyboardInterrupt: 

In [7]:
file_path = r"D:\pipeline-data\citations\datacite\citation_blocks\citation_blocks.ndjson"
batch_find_citations_from_dc_parallel(file_path, out_ndjson, date_parquet)

[*] Loading pubdate cache from datacite_citation_dois_with_pubdates.parquet...
[*] Counting exact lines in input file...
    -> Total workload: 1,565,793 lines.
[*] Launching 32 workers...
Progress: 100.00% | Processed: 1,565,776/1,565,793 lines
[*] Merging 32 part files into final output...
[DONE] Finished in 5958.56s.
       Total Citations Found: 4,641,366
